# 08. parallelize.py: распределяем математику

На одном GPU Mac/Colab процессы не нужны. Разберём parallelize_dit, get_module_by_name, update_plan_for_lora и parallelize_seq через проверяемые CPU-эквиваленты матричных операций.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladislav-vasilenko/generative-ai-research/blob/main/kandinsky-5/notebooks/08_parallelize.ipynb)

### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


### 1. Data, tensor и sequence parallel

Data parallel делит примеры, tensor parallel делит веса/каналы внутри слоя, sequence parallel делит токены. FSDP хранит шарды весов и собирает нужные параметры. Это разные способы уменьшить память и распределить работу; ни один не меняет число обученных параметров.


In [ ]:
source_excerpt('kandinsky/models/parallelize.py','def parallelize_dit',31)


### 2. ColwiseParallel — делим выходные признаки

Для Linear y=xWᵀ+b разделим строки W (выходные каналы). Каждый виртуальный участник вычисляет свою часть y, затем concatenate собирает результат. Название colwise относится к логической матричной операции; PyTorch хранит W в [out,in].


In [ ]:
x=torch.randn(5,6); W=torch.randn(8,6); b=torch.randn(8)
full=x@W.T+b
parts=[x@w.T+bb for w,bb in zip(W.chunk(2,0),b.chunk(2,0))]
assembled=torch.cat(parts,-1)
assert torch.allclose(full,assembled,atol=1e-6)
print('shard outputs:',[p.shape for p in parts])


### 3. RowwiseParallel — складываем частичные суммы

Разделяем входные признаки и соответствующие столбцы W. Частичные результаты суммируются all_reduce. Bias прибавляется один раз после sum; добавление bias каждым участником даст ошибку.


In [ ]:
partial=[xx@ww.T for xx,ww in zip(x.chunk(2,-1),W.chunk(2,-1))]
assembled=sum(partial)+b
assert torch.allclose(full,assembled,atol=1e-6)
print('max difference:',(assembled-full).abs().max().item())


### 4. Sequence parallel и глобальный контекст

FFN/LayerNorm работают независимо по токенам: можно делить sequence и собрать обратно. Self-attention требует доступа к остальным K/V; локальный attention на каждом шарде без обмена ошибочен. Покажем различие на малой задаче.


In [ ]:
import torch.nn.functional as F
q=torch.randn(1,1,8,4); k=torch.randn_like(q); v=torch.randn_like(q)
correct=F.scaled_dot_product_attention(q,k,v)
local=torch.cat([F.scaled_dot_product_attention(qq,kk,vv) for qq,kk,vv in zip(q.chunk(2,2),k.chunk(2,2),v.chunk(2,2))],2)
print('ошибка без global KV:',(correct-local).abs().mean().item())


### 5. LoRA меняет имена вложенных модулей

get_module_by_name проходит dotted path, включая ModuleList indices. update_plan_for_lora переносит parallelization plan на base_layer и adapter layers. Только LoRA A/B можно сделать trainable, базовый DiT оставить frozen. Это способ адаптации, а не часть обязательного T2V инференса.


In [ ]:
tree=torch.nn.Module(); tree.blocks=torch.nn.ModuleList([torch.nn.Linear(3,4)])
def get_module_by_name(root,path):
    for name in path.split('.'): root=getattr(root,name)
    return root
print(get_module_by_name(tree,'blocks.0'))
source_excerpt('kandinsky/models/parallelize.py','def update_plan_for_lora',23)


### 6. parallelize_seq в настоящем коде

Для нескольких GPU используется DeviceMesh и DTensor layouts Shard/Replicate. Не вызываем distributed launch внутри notebook с одной GPU. Даже Colab A100 — это один device; параллелизм нескольких GPU проверяется отдельно на соответствующем сервере.


In [ ]:
source_excerpt('kandinsky/models/parallelize.py','def parallelize_seq',65)


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
